In [1]:
import numpy as np
import pyvista as pv

from phd_helpers.paths import get_subject_stl_path, get_bone_inertia, transform_mesh, get_intercepts

In [ ]:
def compute_x_dist(tpm, mc1, return_points=False, cartilage_id=-2):
    """Compute the unsigned minimum distance in the x direction between the trapezium and metacarpal cartilage surfaces"""
    # extract cartilage surfaces to speed up computation
    mc1_cartilage_surf = mc1.extract_cells(np.where(mc1['region_id']==cartilage_id)[0]).extract_geometry()
    tpm_cartilage_surf = tpm.extract_cells(np.where(tpm['region_id']==cartilage_id)[0]).extract_geometry()
    # compute distance in direction of metacarpal principal axis (x) between cartilage surfaces
    vecs_x = np.zeros_like(mc1_cartilage_surf.points)+np.array([-1, 0, 0])
    points_tpm, points_mc1, mask_points_mc1 = get_intercepts(tpm_cartilage_surf, mc1_cartilage_surf.points, vecs_x)
    dists_x = np.linalg.norm(points_mc1 - points_tpm, axis=1)
    if not return_points:
        return min(dists_x)
    else:
        return min(dists_x), points_tpm, points_mc1

In [20]:
# load subject inertial data
subject, sideL = 14548, 'R'
stl_path = get_subject_stl_path(subject, sideL)
mc1_centroid, _, mc1_axes = get_bone_inertia(stl_path, 'mc1')
tpm_centroid, _, tpm_axes = get_bone_inertia(stl_path, 'tpm')

# load meshes and move origin to tpm centroind align with axes with metacarpal principal axes
tpm = pv.read(f'../volumetricMesh/cgal-box/tpm-final.vtu')
tpm = transform_mesh(tpm, mc1_axes, tpm_centroid, inverse=True)
mc1 = pv.read(f'../volumetricMesh/cgal-box/mc1-final.vtu')
mc1 = transform_mesh(mc1, mc1_axes, tpm_centroid, inverse=True)

def position_mc1_tpm(mc1, tpm, target_dist=0.01, raise_error=False):
    """Given mc1 and tpm (with cartilage) in mc1 inertial basis 
    - moves tpm so that closest points on cartilage surfaces are target_dist apart in x direction\n
    Doesn't copy meshes - moves them inplace"""

    # move tpm into position
    tpm.points -= np.array([10, 0, 0]) # ensure no interference before computing x-distance
    min_dist_x = compute_x_dist(tpm, mc1)
    move_dist = min_dist_x - target_dist
    tpm.points += np.array([move_dist, 0, 0])

    # checks
    final_dist = abs(compute_x_dist(tpm, mc1))
    print(f"Distance between cartilage surfaces (x->): {final_dist:.4f}")
    interference_check = (tpm.extract_surface().compute_implicit_distance(mc1.extract_surface())['implicit_distance'] >= 0).all()
    print("No interference: ", interference_check)

    if raise_error:
        if not interference_check:
            raise RuntimeError(f"Bones not positioned correctly - Interference found")
        if round(final_dist, 4) != target_dist:
            raise RuntimeError(f"Bones not positioned correctly - final dist = {final_dist:.4f}")

position_mc1_tpm(mc1, tpm, target_dist=0.01, raise_error=True)

# save
#tpm.save(f'../volumetricMesh/cgal-box/tpm-final-postioned.vtu')
#mc1.save(f'../volumetricMesh/cgal-box/mc1-final-postioned.vtu')

Distance between cartilage surfaces (x->): 0.0100
No interference:  True


In [13]:
pl = pv.Plotter()
pl.add_mesh(tpm, scalars='region_id')
pl.add_mesh(mc1, scalars='region_id')
pl.show()

Widget(value='<iframe src="http://localhost:55654/index.html?ui=P_0x3131d90d0_1&reconnect=auto" class="pyvista…